# Installations

In [ ]:
# # installing offline dependencies
# !pip install -U /kaggle/input/faiss-gpu-173-python310/faiss_gpu-1.7.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl
# !cp -rf /kaggle/input/sentence-transformers-222/sentence-transformers /kaggle/working/sentence-transformers
# !pip install -U /kaggle/working/sentence-transformers
# !pip install -U /kaggle/input/blingfire-018/blingfire-0.1.8-py3-none-any.whl

# !pip install --no-index --no-deps /kaggle/input/llm-whls/transformers-4.31.0-py3-none-any.whl
# !pip install --no-index --no-deps /kaggle/input/llm-whls/peft-0.4.0-py3-none-any.whl
# !pip install --no-index --no-deps /kaggle/input/llm-whls/datasets-2.14.3-py3-none-any.whl
# !pip install --no-index --no-deps /kaggle/input/llm-whls/trl-0.5.0-py3-none-any.whl

# Load libraries

In [ ]:
# import os, time
# import gc
# import pandas as pd
# import numpy as np
# import re
# from tqdm.auto import tqdm
# import blingfire as bf
# from __future__ import annotations

# from collections.abc import Iterable

# import faiss
# from faiss import write_index, read_index

# from sentence_transformers import SentenceTransformer

# import torch
# import ctypes
# libc = ctypes.CDLL("libc.so.6")

# from dataclasses import dataclass
# from typing import Optional, Union

# import torch
# import numpy as np
# import pandas as pd
# from datasets import Dataset
# from transformers import AutoTokenizer
# from transformers import AutoModelForMultipleChoice, TrainingArguments, Trainer
# from transformers.tokenization_utils_base import PreTrainedTokenizerBase, PaddingStrategy
# from torch.utils.data import DataLoader

# from scipy.special import softmax

In [ ]:
import os, time
import gc
import pandas as pd
import numpy as np
import re
from tqdm.auto import tqdm

In [ ]:
from scipy.optimize import minimize, fsolve
import datetime
import torch.nn.functional as F
from numba import njit

# Custom functions

In [ ]:
def apk(actual, predicted, k=5):
    """
    Computes the average precision at k.
    This function computes the average prescision at k between two lists of
    items.
    Parameters
    ----------
    actual : list
             A list of elements that are to be predicted (order doesn't matter)
    predicted : list
                A list of predicted elements (order does matter)
    k : int, optional
        The maximum number of predicted elements
    Returns
    -------
    score : double
            The average precision at k over the input lists
    """
    
    # requires all elements are unique
    assert (len(np.unique(predicted)) == len(predicted))

    if len(predicted)>k:
        predicted = predicted[:k]

    score = 0.0
    num_hits = 0.0

    for i,p in enumerate(predicted):
        # first condition checks whether it is valid prediction
        # second condition checks if prediction is not repeated
        if p in actual and p not in predicted[:i]:
            num_hits += 1.0
            score += num_hits / (i + 1.0)

    return score / min(len(actual), k)

def mapk(actual, predicted, k=5):
    
    """
    Computes the mean average precision at k.
    This function computes the mean average prescision at k between two lists
    of lists of items.
    Parameters
    ----------
    actual : list
             A list of lists of elements that are to be predicted 
             (order doesn't matter in the lists)
    predicted : list
                A list of lists of predicted elements
                (order matters in the lists)
    k : int, optional
        The maximum number of predicted elements
    Returns
    -------
    score : double
            The mean average precision at k over the input lists
    """
    return np.mean([apk(a,p,k) for a,p in zip(actual, predicted)])

In [ ]:
@njit
def grad_func_jit(weights):
    preds_clip = np.minimum(1 - 1e-15, np.maximum(preds, 1e-15))
    gradients = np.zeros(preds.shape[0])
    for i in range(preds.shape[0]):
        a, b, c = target_values, preds_clip[i], np.zeros((preds.shape[1], preds.shape[2]))
        a = np.eye(5)[a]
        for j in range(preds.shape[0]):
            if j != i:
                c += weights[j] * preds_clip[j]
        gradients[i] = -np.mean((-a*b+(b**2)*weights[i]+b*c)/((b**2)*(weights[i]**2)+2*b*c*weights[i]-b*weights[i]+(c**2)-c))
    return gradients

In [ ]:
def calc_mtr(predicted, k=3):
    y_preds = np.argsort(-predicted, 1)
    map3 = mapk(target_values.reshape(-1, 1), y_preds.reshape(-1, 5), k=k)
    return map3

# def calc_loss(predicted):
#     score = F.cross_entropy(torch.tensor(predicted), torch.tensor(target_values)).numpy()
#     return score


def log_loss_numpy(y_pred):
    y_pred = np.clip(y_pred, 1e-15, 1 - 1e-15)
    loss = - target_values_one_hot * np.log(y_pred)
    loss = np.sum(loss, axis=-1)
    return loss.mean()

def func_to_optimise(weights):
    pred_blend = np.tensordot(weights, preds, axes = ((0), (0)))
    score = log_loss_numpy(pred_blend)
    return score

def func_to_map3(weights):
    pred_blend = np.tensordot(weights, preds, axes = ((0), (0)))
    score = calc_mtr(pred_blend)
    return score

# Read data

In [ ]:
val_df = pd.read_csv('/kaggle/input/llmse-eval/LLMSE_1k.csv')
val_df.reset_index(inplace=True, drop=True)
val_df.shape

In [ ]:
options = 'ABCDE'
indices = list(range(5))

option_to_index = {option: index for option, index in zip(options, indices)}
index_to_option = {index: option for option, index in zip(options, indices)}
target_values = val_df['answer'].map(option_to_index).values
target_values_one_hot = np.eye(5)[target_values]

# Get Val prediction arrays

In [ ]:
PRED_LOC = "/kaggle/input/llmse-1kvalpreds/"

preds_dict = dict()

for file in os.listdir(PRED_LOC):
    
    modelnm = file.split('_')[0]
    
    globals()[f'{modelnm}_df'] = pd.read_csv(PRED_LOC+file)
    
    keepCols = ['A_prob', 'B_prob', 'C_prob', 'D_prob', 'E_prob']
    
    preds_dict[modelnm] = np.array(globals()[f'{modelnm}_df'][keepCols])
    
    print(f"{modelnm}: ", preds_dict[modelnm].shape)
    

# Prediction scores

In [ ]:
preds = np.zeros((len(preds_dict), len(val_df), 5))
for i in range(preds.shape[0]):
    preds[i] = list(preds_dict.values())[i]

In [ ]:
%%time

map3_scores = {}
for n, key in enumerate(preds_dict.keys()):
    score_val = calc_mtr(preds[n])
    map3_scores[key] = score_val
    print(f'{key:40s} CV_map@3:', score_val)
    
print('-' * 60)

loss_scores = {}
for n, key in enumerate(preds_dict.keys()):
    score_val = log_loss_numpy(preds[n])
    loss_scores[key] = score_val
    print(f'{key:40s} CV_CELoss:', score_val)
    
print('-' * 60)

ln(5) = 1.60943791243 and losses are about 1.5, So the models seem to be making near-random predictions. As I said at the beginning, the validation dataset may not be suitable, but I will continue.

# Correlation

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.style as style
import seaborn as sns
from matplotlib import pyplot
from matplotlib.ticker import ScalarFormatter
sns.set_context("talk")
style.use('fivethirtyeight')

subs = np.zeros((len(preds_dict), len(val_df), 5))

for i, p in enumerate(preds_dict.keys()):
    print(i,p)
    subs[i,:,:] = list(preds_dict.values())[i]
    
corr = np.corrcoef(subs.reshape(len(preds_dict), -1))

# Set up the matplotlib figure
f, ax = plt.subplots(figsize=(15, 12))

# Generate a custom diverging colormap
cmap = sns.diverging_palette(220, 10, as_cmap=True)

# Draw the heatmap with the mask and correct aspect ratio
sns.heatmap(corr, cmap=cmap, annot=True, fmt="g",
            square=True, linewidths=.5, cbar_kws={"shrink": .5}, ax=ax)
ax.set_ylim(corr.shape[0], 0)
plt.yticks(rotation=0)

## Hill Climb

In [ ]:
import logging

logging.basicConfig(filename="./HillClimbPreds.log", level=logging.INFO)

In [ ]:
CONFIG = {
        'SEED': 101,
        'RES': 20,
        'PATIENCE': 1
        }

models = list(preds_dict.keys())

In [ ]:
models

In [ ]:
log_df = pd.DataFrame()

roundno = []
start = []
addM = []
weights = []
metric = []

for i in tqdm(range(len(models))):
    
    champ = 0
    chall = 0
    
    preds = preds_dict[models[i]]
    
    logging.info("-"*25)
    logging.info(f"{models[i]}")
    logging.info("-"*25)
    
    hillclimb = 1
    addModel = ''
    wtadd = 1
    
    while True:
        
        champ = calc_mtr(preds)
        
        logging.info(f"Round {hillclimb}")
    
        logging.info(f"Score = {champ}")
            
        roundno.append(hillclimb)
        start.append(models[i])
        addM.append(addModel)
        weights.append(wtadd)
        metric.append(champ)
        
        addModel = ''
        wtadd = 1
        
        for model in models:
            for wt in range(CONFIG['RES']):

                temp = ((wt/CONFIG['RES']) * preds) + ((1-wt/CONFIG['RES']) * preds_dict[model])
                chall = calc_mtr(temp)
            
                if chall > champ:
                    champ = chall
                    wtadd = (1-wt/CONFIG['RES'])
                    addModel = model
                    newpred = temp
                
        
        if addModel=='':
            logging.info("Metric has not improved")
            _ = gc.collect()
            break
        else:
            logging.info(f"Adding {addModel} with weight {wtadd}")
            preds = newpred
            logging.info("-"*25)
            hillclimb += 1
            

log_df['round'] = roundno 
log_df['start'] = start 
log_df['add'] = addM 
log_df['weight'] = weights 
log_df['metric'] = metric 

log_df.to_csv("HillClimbPreds_log.csv", index=False)

In [ ]:
print("Done")